In [1]:
import kagglehub
import pandas as pd
import numpy as np
import json

/Users/shauryasingru/Desktop/ML Projects/movie-recommender-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download(
                            "tmdb/tmdb-movie-metadata",
                            output_dir = '../data')
movies_path =  path + "/tmdb_5000_movies.csv"
credits_path = path + "/tmdb_5000_credits.csv"

In [3]:
movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

In [188]:
movies = movies[movies['original_language'] == 'en']

In [189]:
movie_keywords = {}
movie_genres = {}

for row in movies.iterrows():
    row = pd.Series(row)
    name = row[1]['original_title']
    keywords_list = json.loads(row[1]['keywords'])
    genres_list = json.loads(row[1]['genres'])

    keyword_names = []

    for keyword in keywords_list:
        keyword_names.append(keyword['name'].replace(' ', ''))

    movie_keywords[name] = keyword_names

    genre_names = []

    for genre in genres_list:
        genre_names.append(genre['name'].lower().replace(' ', ''))

    movie_genres[name] = genre_names



In [190]:
movie_cast = {}
movie_directors = {}

for row in credits.iterrows():
    row = pd.Series(row)
    name = row[1]['title']
    cast_list = json.loads(row[1]['cast'])
    crew_list = json.loads(row[1]['crew'])

    actor_names = []

    for actor in cast_list:
        actor_names.append(actor['name'].lower().replace(' ', ''))

    movie_cast[name] = actor_names

    for crew_member in crew_list:
        if crew_member['department'] == 'Directing':
            movie_directors[name] = crew_member['name'].lower().replace(' ', '')

In [191]:
soup_df = pd.DataFrame() # Initialising an empty df

for movie in movies.original_title:
    
    try:
        director = movie_directors[movie]
    except:
        director = ''

    try:
        cast = movie_cast[movie]
    except:
        cast = ''
    
    row = {
        'title': [movie],
        'cast': [cast],
        'keywords': [movie_keywords[movie]],
        'genres': [movie_genres[movie]],
        'director': [director]
    }

    soup_df = pd.concat([soup_df, pd.DataFrame(row)], axis = 0)

soup_df = soup_df.reset_index(drop=True)
soup_df

,title,cast,keywords,genres,director
0,Avatar,"[samworthington, zoesaldana, sigourneyweaver, ...","[cultureclash, future, spacewar, spacecolony, ...","[action, adventure, fantasy, sciencefiction]",jamescameron
1,Pirates of the Caribbean: At World's End,"[johnnydepp, orlandobloom, keiraknightley, ste...","[ocean, drugabuse, exoticisland, eastindiatrad...","[adventure, fantasy, action]",karengolden
2,Spectre,"[danielcraig, christophwaltz, léaseydoux, ralp...","[spy, basedonnovel, secretagent, sequel, mi6, ...","[action, adventure, crime]",susiejones
3,The Dark Knight Rises,"[christianbale, michaelcaine, garyoldman, anne...","[dccomics, crimefighter, terrorist, secretiden...","[action, crime, drama, thriller]",nilootero
4,John Carter,"[taylorkitsch, lynncollins, samanthamorton, wi...","[basedonnovel, mars, medallion, spacetravel, p...","[action, adventure, sciencefiction]",andrewm.ward
...,...,...,...,...,...
4500,Cavite,[],[],"[foreign, thriller]",iangamazon
4501,Newlyweds,"[edwardburns, kerrybishé, marshadietlein, cait...",[],"[comedy, romance]",edwardburns
4502,"Signed, Sealed, Delivered","[ericmabius, kristinbooth, crystallowe, geoffg...","[date, loveatfirstsight, narration, investigat...","[comedy, drama, romance, tvmovie]",scottsmith
4503,Shanghai Calling,"[danielhenney, elizacoupe, billpaxton, alanruc...",[],[],danielhsia


In [192]:
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])
soup_df['soup'] = soup_df.apply(create_soup, axis=1)
soup_df = soup_df.set_index('title').sort_index(ascending=True)

In [194]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(soup_df['soup'])

In [195]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [234]:
movies.sort_values(by = 'original_title').reset_index(drop=True)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,0,"[{""id"": 53, ""name"": ""Thriller""}]",http://supercapitalist.net/,119458,[],en,$upercapitalist,A maverick New York hedge fund trader with unc...,0.174311,[],"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-08-10,0,103.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Money for Life,Supercapitalist,3.5,2
1,7500000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",http://500days.com,19913,"[{""id"": 248, ""name"": ""date""}, {""id"": 572, ""nam...",en,(500) Days of Summer,"Tom (Joseph Gordon-Levitt), greeting-card writ...",45.610993,"[{""name"": ""Fox Searchlight Pictures"", ""id"": 43...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-07-17,60722734,95.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,It was almost like falling in love.,(500) Days of Summer,7.2,2904
2,15000000,"[{""id"": 53, ""name"": ""Thriller""}, {""id"": 878, ""...",http://www.10cloverfieldlane.com/,333371,"[{""id"": 1930, ""name"": ""kidnapping""}, {""id"": 23...",en,10 Cloverfield Lane,"After a car accident, Michelle awakens to find...",53.698683,"[{""name"": ""Paramount Pictures"", ""id"": 4}, {""na...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2016-03-10,108286421,103.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Monsters come in many forms.,10 Cloverfield Lane,6.8,2468
3,1200000,"[{""id"": 18, ""name"": ""Drama""}]",NaN,345003,"[{""id"": 1568, ""name"": ""undercover""}, {""id"": 49...",en,10 Days in a Madhouse,"Nellie Bly, a 23 year-old reporter for Joseph ...",0.489271,[],"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2015-11-20,0,111.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,10 Days in a Madhouse,4.3,5
4,16000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",NaN,4951,"[{""id"": 497, ""name"": ""shakespeare""}, {""id"": 59...",en,10 Things I Hate About You,"Bianca, a tenth grader, has never gone on a da...",54.550275,"[{""name"": ""Mad Chance"", ""id"": 1757}, {""name"": ...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1999-03-30,53478166,97.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,How do I loathe thee? Let me count the ways.,10 Things I Hate About You,7.3,1701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4500,15000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 53, ""nam...",NaN,1946,"[{""id"": 282, ""name"": ""video game""}, {""id"": 215...",en,eXistenZ,A game designer on the run from assassins must...,21.928025,"[{""name"": ""Alliance Atlantis Communications"", ...","[{""iso_3166_1"": ""CA"", ""name"": ""Canada""}, {""iso...",1999-04-14,2856712,97.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Play it. Live it. Kill for it.,eXistenZ,6.7,475
4501,70000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/xxx/,7451,"[{""id"": 999, ""name"": ""sports car""}, {""id"": 186...",en,xXx,Xander Cage is your standard adrenaline junkie...,46.217769,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2002-08-09,277448382,124.0,"[{""iso_639_1"": ""cs"", ""name"": ""\u010cesk\u00fd""...",Released,A New Breed Of Secret Agent.,xXx,5.8,1424
4502,60000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/xxxstateoft...,11679,"[{""id"": 521, ""name"": ""washington d.c.""}, {""id""...",en,xXx: State of the Union,"Ice Cube stars as Darius Stone, a thrill-seeki...",36.689223,"[{""name"": ""Original Film"", ""id"": 333}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2005-04-27,71073932,101.0,"[{""iso_639_1

In [232]:
def get_recommendations(title):
    movies_df = movies.sort_values(by = 'original_title').reset_index(drop=True)
    if any(movies_df['original_title'].isin([title])):
        idx = movies_df.index[movies['original_title'] == title][0]
    else:
        return 'Movie not found. Check again'
    
    scores = list(enumerate(cosine_sim[idx]))

    scores = sorted(scores, key = lambda x: x[1], reverse = True)

    scores = scores[0:11]

    movie_idx = [i[0] for i in scores]

    return movies_df.loc[movie_idx ,'original_title'].reset_index(drop=True)
        
    

In [228]:
soup_df.loc['The Dark Knight Rises']

cast        [christianbale, michaelcaine, garyoldman, anne...
keywords    [dccomics, crimefighter, terrorist, secretiden...
genres                       [action, crime, drama, thriller]
director                                            nilootero
soup        dccomics crimefighter terrorist secretidentity...
Name: The Dark Knight Rises, dtype: object

In [229]:
soup_df.loc['Cama adentro']

cast                        
keywords                  []
genres      [drama, foreign]
director                    
soup           drama foreign
Name: Cama adentro, dtype: object

In [238]:
get_recommendations('La femme de chambre du Titanic')

0                          Soul Men
1         Down & Out With The Dolls
2     Jackass Presents: Bad Grandpa
3                       Roll Bounce
4      Michael Jackson's This Is It
5                      Next Day Air
6       Welcome Home Roscoe Jenkins
7                      Johnny Suede
8                      The Best Man
9                Undercover Brother
10     The Original Kings of Comedy
Name: original_title, dtype: object

In [169]:
def get_recommendations(title, cosine_sim=cosine_sim):
    # Get the index of the movie that matches the title
    idx = indices[title]

    # Get the pairwsie similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar movies
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar movies
    return df2['title'].iloc[movie_indices]